# **V. Análisis de variables de tipo fecha**

## Objetivo
Este notebook tiene el fin de analizar la calidad, formato y rango de las variables tipo fecha presentes en el dataset clasificado del GRD (2019-2024), identificando fechas válidas, nulos reales, valores inválidos y patrones temporales a nivel sistémico para garantizar consistencia en el procesamiento de datos.

## Proceso
1. **Configuración y Carga Optimizada**: Especificación de rutas y lectura selectiva en memoria RAM para datasets masivos.
2. **Limpieza de fechas**: Identificación y separación de los nulos reales de valores problemáticos.
3. **Detección de formatos**: Reconocimiento de múltiples formatos de fecha (ISO, dd-mm-yyyy, yyyymmdd).
4. **Parseo robusto**: Conversión de strings a datetime con múltiples intentos de parsing.
5. **Análisis de calidad**: Evaluar validez, rango temporal y distribución de anomalías.
6. **Generación de reportes**: Exportar matrices de análisis general por año y variable de fecha.

### Columnas de la matriz de análisis:

| Columna | Significado |
|---------|-------------|
| **Año** | Año del análisis (2019-2024) |
| **Columna** | Nombre de la variable de fecha analizada |
| **Formato** | Formato detectado (ISO, dd-mm-yyyy, yyyymmdd, desconocido) |
| **Total** | Número total de registros en el archivo |
| **NULL reales** | Valores realmente nulos (NaN de pandas) |
| **% NULL** | Porcentaje de nulos reales |
| **Inválidos** | Strings que no pudieron parsearse como fecha |
| **% Inválidos** | Porcentaje de fechas inválidas |
| **% Válidos** | Porcentaje de fechas válidas (parseadas exitosamente) |
| **Año min/max** | Rango de años en las fechas válidas |
| **Mes min/max** | Rango de meses (1-12) |
| **Día min/max** | Rango de días del mes (1-31) |

### Cómo interpretar:
- **% Válidos cercano a 100%**: Variable de buena calidad
- **% Inválidos > 5%**: Revisar formatos o limpieza de datos
- **Rango de años sospechoso**: Posibles errores de entrada (ej: año 2099, 1899)
- **Mes/Día fuera de rango**: Indicativo de formato no detectado o corrupción

In [1]:
# ============================================================================
# CONFIGURACIÓN INICIAL: Imports y Rutas
# ============================================================================

import pandas as pd
import os
import re
from collections import Counter
import warnings

# Suprimir advertencias menores de parsing
warnings.filterwarnings('ignore', category=UserWarning)

# Ruta de datos MASIVOS clasificados
carpeta = "../../Datos/Datos clasificados"

# Ruta de resultados para guardar análisis de fechas
resultados_fecha = "../../Resultados/Resultados (etapa 1 y 2)/Análisis de fechas"
os.makedirs(resultados_fecha, exist_ok=True)

# Lista de archivos a procesar (años 2019-2024)
archivos = [
    "GRD_CLASIFICADO_2019.csv",
    "GRD_CLASIFICADO_2020.csv",
    "GRD_CLASIFICADO_2021.csv",
    "GRD_CLASIFICADO_2022.csv",
    "GRD_CLASIFICADO_2023.csv",
    "GRD_CLASIFICADO_2024.csv"
]

# Mapeo de equivalencias para estandarizar nombres de columnas inconsistentes
mapa_columnas = {
    "ï»¿COD_HOSPITAL": "COD_HOSPITAL",
    "ID_BENEFICIARIO": "CIP_ENCRIPTADO"
}

# ============================================================================
# UTILIDADES: Funciones de Limpieza y Parseo de Fechas
# ============================================================================

def limpiar_columna_fecha(col):
    col = col.astype(str).str.strip()
    col = col.mask(col.isin([
        "", "nan", "NaN", "None", "NULL",
        "0000-00-00", "00000000"
    ]), None)
    return col

def detectar_formato(ejemplos):
    formatos = []
    for val in ejemplos:
        if re.match(r"^\d{4}-\d{2}-\d{2}$", val):
            formatos.append("yyyy-mm-dd")
        elif re.match(r"^\d{2}-\d{2}-\d{4}$", val):
            formatos.append("dd-mm-yyyy")
        elif re.match(r"^\d{8}$", val):
            formatos.append("yyyymmdd")

    if formatos:
        return Counter(formatos).most_common(1)[0][0]
    return "desconocido"

def analizar_fecha_tabla(carpeta, archivos, mapa_columnas, columna_fecha):
    """
    Genera un análisis de calidad de fechas leyendo masivamente pero de 
    forma selectiva en RAM para evitar colapsos de memoria.
    """
    print(f"\n[{columna_fecha}] Iniciando escaneo de calidad temporal...")
    resultados = []

    for archivo in archivos:
        ruta = os.path.join(carpeta, archivo)
        año_archivo = archivo[-8:-4]
        
        # 1. OPTIMIZACIÓN: Solo leer encabezados primero
        encabezados = pd.read_csv(ruta, nrows=0).rename(columns=mapa_columnas).columns
        if columna_fecha not in encabezados:
            print(f"  -> Advertencia: {año_archivo}: Columna '{columna_fecha}' no encontrada")
            continue
            
        # 2. OPTIMIZACIÓN: Leer SOLO la columna requerida
        columnas_a_leer = [col for col in pd.read_csv(ruta, nrows=0).columns 
                           if col == columna_fecha or mapa_columnas.get(col) == columna_fecha]
        
        df = pd.read_csv(ruta, usecols=columnas_a_leer, low_memory=False).rename(columns=mapa_columnas)
        total = len(df)

        col_original = df[columna_fecha]
        mask_null_original = col_original.isna()
        
        col = col_original.copy().astype("string")
        col[~mask_null_original] = col[~mask_null_original].str.strip()
        col[mask_null_original] = None
        col = col.mask(col.isin(["nan", "NaN", "None", "NULL"]), None)

        # Detección de Formato
        ejemplos = col.dropna().head(50).tolist()
        formato_detectado = detectar_formato(ejemplos)

        # Parseo Robusto
        f1 = pd.to_datetime(col, errors="coerce", format="%Y-%m-%d")
        mask_no_iso = ~col.str.match(r"^\d{4}-\d{2}-\d{2}$", na=False)
        f2 = pd.to_datetime(col.where(mask_no_iso), errors="coerce", dayfirst=True)
        fechas = f1.fillna(f2)

        # Clasificación
        mask_null_real = col.isna()
        mask_invalidos = fechas.isna() & col.notna()

        n_null_real = mask_null_real.sum()
        n_invalidos = mask_invalidos.sum()
        n_total_nulos = fechas.isna().sum()

        pct_null_real = (n_null_real / total) * 100
        pct_invalidos = (n_invalidos / total) * 100
        pct_validos = 100 - ((n_total_nulos / total) * 100)

        # Rangos
        años = fechas.dt.year.dropna()
        meses = fechas.dt.month.dropna()
        dias = fechas.dt.day.dropna()

        resultados.append({
            "Año": int(año_archivo),
            "Columna": columna_fecha,
            "Formato": formato_detectado,
            "Total": total,
            "NULL reales": n_null_real,
            "% NULL": round(pct_null_real, 2),
            "Inválidos": n_invalidos,
            "% Inválidos": round(pct_invalidos, 2),
            "% Válidos": round(pct_validos, 2),
            "Año min": int(años.min()) if len(años) > 0 else None,
            "Año max": int(años.max()) if len(años) > 0 else None,
            "Mes min": int(meses.min()) if len(meses) > 0 else None,
            "Mes max": int(meses.max()) if len(meses) > 0 else None,
            "Día min": int(dias.min()) if len(dias) > 0 else None,
            "Día max": int(dias.max()) if len(dias) > 0 else None,
        })

    df_resultado = pd.DataFrame(resultados).sort_values("Año")
    
    ruta_salida = os.path.join(resultados_fecha, f"{columna_fecha.lower()}_analisis_fechas.csv")
    df_resultado.to_csv(ruta_salida, index=False, encoding="utf-8-sig")
    print(f"Análisis guardado en: {ruta_salida}")
    
    return df_resultado

### **1. Fecha de Nacimiento y Fecha de Ingreso (FECHA_NACIMIENTO, FECHA_INGRESO)**
- **Tipo de variables**: Demográfica y Hospitalaria.
- **Descripción**: Indican el día exacto de nacimiento del paciente y el día de inicio del episodio asistencial.
- **Análisis de Calidad**: Excelente. Ambas presentan un 0% de nulos reales en toda la cohorte. Se detectó un quiebre de formato sistémico en 2023 (cambio de `yyyy-mm-dd` a `dd-mm-yyyy`), el cual fue resuelto exitosamente mediante un parseo robusto.
- **Veredicto**: **Descartar columnas crudas y Transformar en Variable Derivada (`EDAD_INGRESO`)**. Aunque los algoritmos basados en árboles (Random Forest, XGBoost) no procesan objetos *datetime* de forma nativa, la combinación de ambas fechas es vital. La diferencia matemática entre ambas generará la variable `EDAD_INGRESO` (en años cumplidos). La edad es el determinante demográfico de riesgo más importante en oncología clínica, por lo que su inclusión es crítica para el perfilamiento.

In [2]:
analizar_fecha_tabla(carpeta, archivos, mapa_columnas, "FECHA_NACIMIENTO")


[FECHA_NACIMIENTO] Iniciando escaneo de calidad temporal...
Análisis guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de fechas\fecha_nacimiento_analisis_fechas.csv


,Año,Columna,Formato,Total,NULL reales,% NULL,Inválidos,% Inválidos,% Válidos,Año min,Año max,Mes min,Mes max,Día min,Día max
0,2019,FECHA_NACIMIENTO,yyyy-mm-dd,1151398,0,0.0,16,0.0,100.0,1894,2019,1,12,1,31
1,2020,FECHA_NACIMIENTO,yyyy-mm-dd,781911,0,0.0,1,0.0,100.0,1900,2020,1,12,1,31
2,2021,FECHA_NACIMIENTO,yyyy-mm-dd,816909,0,0.0,0,0.0,100.0,1900,2021,1,12,1,31
3,2022,FECHA_NACIMIENTO,yyyy-mm-dd,932837,0,0.0,7,0.0,100.0,1912,2022,1,12,1,31
4,2023,FECHA_NACIMIENTO,yyyy-mm-dd,1039587,0,0.0,10,0.0,100.0,1914,2023,1,12,1,31
5,2024,FECHA_NACIMIENTO,yyyy-mm-dd,1085813,0,0.0,3,0.0,100.0,1915,2024,1,12,1,31


In [3]:
analizar_fecha_tabla(carpeta, archivos, mapa_columnas, "FECHA_INGRESO")


[FECHA_INGRESO] Iniciando escaneo de calidad temporal...
Análisis guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de fechas\fecha_ingreso_analisis_fechas.csv


,Año,Columna,Formato,Total,NULL reales,% NULL,Inválidos,% Inválidos,% Válidos,Año min,Año max,Mes min,Mes max,Día min,Día max
0,2019,FECHA_INGRESO,yyyy-mm-dd,1151398,0,0.0,18,0.0,100.0,2012,2019,1,12,1,31
1,2020,FECHA_INGRESO,yyyy-mm-dd,781911,0,0.0,0,0.0,100.0,2011,2020,1,12,1,31
2,2021,FECHA_INGRESO,yyyy-mm-dd,816909,0,0.0,0,0.0,100.0,2012,2021,1,12,1,31
3,2022,FECHA_INGRESO,yyyy-mm-dd,932837,0,0.0,0,0.0,100.0,2013,2022,1,12,1,31
4,2023,FECHA_INGRESO,dd-mm-yyyy,1039587,0,0.0,0,0.0,100.0,2022,2023,1,12,1,31
5,2024,FECHA_INGRESO,yyyy-mm-dd,1085813,0,0.0,52,0.0,100.0,2023,2024,1,12,1,31


### **2. Fechas de Traslado (FECHATRASLADO1 al 9)**
- **Tipo de variables**: Hospitalaria.
- **Descripción**: Fechas exactas de movimiento intrahospitalario entre servicios clínicos.
- **Análisis de Calidad**: Deficiente. Poseen un déficit de datos masivo. La primera fecha de traslado (`FECHATRASLADO1`) ronda el 81-85% de nulos, escalando rápidamente hasta un 99-100% de nulos a partir de la cuarta fecha.
- **Veredicto**: **Descartar**. Si bien el concepto de "fuga de datos temporal" no aplica en este modelo de caracterización, estas fechas no aportan valor predictivo directo. El esfuerzo clínico y la inestabilidad que representan los traslados ya serán capturados de forma mucho más limpia y densa por la variable derivada `CANTIDAD_TRASLADOS` (calculada previamente). Mantener las fechas crudas solo introduciría matrices vacías que no aportan capacidad de generalización.

In [4]:
analizar_fecha_tabla(carpeta, archivos, mapa_columnas, "FECHATRASLADO1")


[FECHATRASLADO1] Iniciando escaneo de calidad temporal...
Análisis guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de fechas\fechatraslado1_analisis_fechas.csv


,Año,Columna,Formato,Total,NULL reales,% NULL,Inválidos,% Inválidos,% Válidos,Año min,Año max,Mes min,Mes max,Día min,Día max
0,2019,FECHATRASLADO1,yyyy-mm-dd,1151398,985066,85.55,18,0.0,14.44,2014,2019,1,12,1,31
1,2020,FECHATRASLADO1,yyyy-mm-dd,781911,617599,78.99,0,0.0,21.01,2011,2020,1,12,1,31
2,2021,FECHATRASLADO1,yyyy-mm-dd,816909,648637,79.40,0,0.0,20.60,2014,2021,1,12,1,31
3,2022,FECHATRASLADO1,yyyy-mm-dd,932837,770865,82.64,0,0.0,17.36,2016,2022,1,12,1,31
4,2023,FECHATRASLADO1,dd-mm-yyyy,1039587,872748,83.95,0,0.0,16.05,2022,2023,1,12,1,31
5,2024,FECHATRASLADO1,yyyy-mm-dd,1085813,912553,84.04,0,0.0,15.96,2019,2024,1,12,1,31


In [5]:
analizar_fecha_tabla(carpeta, archivos, mapa_columnas, "FECHATRASLADO2")


[FECHATRASLADO2] Iniciando escaneo de calidad temporal...
Análisis guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de fechas\fechatraslado2_analisis_fechas.csv


,Año,Columna,Formato,Total,NULL reales,% NULL,Inválidos,% Inválidos,% Válidos,Año min,Año max,Mes min,Mes max,Día min,Día max
0,2019,FECHATRASLADO2,yyyy-mm-dd,1151398,1097710,95.34,8,0.0,4.66,2014,2019,1,12,1,31
1,2020,FECHATRASLADO2,yyyy-mm-dd,781911,724137,92.61,1,0.0,7.39,2011,2020,1,12,1,31
2,2021,FECHATRASLADO2,yyyy-mm-dd,816909,758749,92.88,0,0.0,7.12,2014,2021,1,12,1,31
3,2022,FECHATRASLADO2,yyyy-mm-dd,932837,881291,94.47,0,0.0,5.53,2016,2022,1,12,1,31
4,2023,FECHATRASLADO2,dd-mm-yyyy,1039587,984495,94.70,0,0.0,5.30,2022,2023,1,12,1,31
5,2024,FECHATRASLADO2,yyyy-mm-dd,1085813,1028719,94.74,0,0.0,5.26,2019,2024,1,12,1,31


In [6]:
analizar_fecha_tabla(carpeta, archivos, mapa_columnas, "FECHATRASLADO3")


[FECHATRASLADO3] Iniciando escaneo de calidad temporal...
Análisis guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de fechas\fechatraslado3_analisis_fechas.csv


,Año,Columna,Formato,Total,NULL reales,% NULL,Inválidos,% Inválidos,% Válidos,Año min,Año max,Mes min,Mes max,Día min,Día max
0,2019,FECHATRASLADO3,yyyy-mm-dd,1151398,1135171,98.59,1,0.0,1.41,2015,2020,1,12,1,31
1,2020,FECHATRASLADO3,yyyy-mm-dd,781911,762920,97.57,0,0.0,2.43,2011,2020,1,12,1,31
2,2021,FECHATRASLADO3,yyyy-mm-dd,816909,797687,97.65,0,0.0,2.35,2014,2021,1,12,1,31
3,2022,FECHATRASLADO3,yyyy-mm-dd,932837,917047,98.31,0,0.0,1.69,2017,2022,1,12,1,31
4,2023,FECHATRASLADO3,dd-mm-yyyy,1039587,1023334,98.44,0,0.0,1.56,2022,2023,1,12,1,31
5,2024,FECHATRASLADO3,yyyy-mm-dd,1085813,1068683,98.42,0,0.0,1.58,2020,2024,1,12,1,31


In [7]:
analizar_fecha_tabla(carpeta, archivos, mapa_columnas, "FECHATRASLADO4")


[FECHATRASLADO4] Iniciando escaneo de calidad temporal...
Análisis guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de fechas\fechatraslado4_analisis_fechas.csv


,Año,Columna,Formato,Total,NULL reales,% NULL,Inválidos,% Inválidos,% Válidos,Año min,Año max,Mes min,Mes max,Día min,Día max
0,2019,FECHATRASLADO4,yyyy-mm-dd,1151398,1145762,99.51,1,0.0,0.49,2015,2019,1,12,1,31
1,2020,FECHATRASLADO4,yyyy-mm-dd,781911,775367,99.16,0,0.0,0.84,2011,2020,1,12,1,31
2,2021,FECHATRASLADO4,yyyy-mm-dd,816909,809645,99.11,0,0.0,0.89,2015,2021,1,12,1,31
3,2022,FECHATRASLADO4,yyyy-mm-dd,932837,927097,99.38,0,0.0,0.62,2018,2022,1,12,1,31
4,2023,FECHATRASLADO4,dd-mm-yyyy,1039587,1033767,99.44,0,0.0,0.56,2022,2023,1,12,1,31
5,2024,FECHATRASLADO4,yyyy-mm-dd,1085813,1079472,99.42,0,0.0,0.58,2020,2024,1,12,1,31


In [8]:
analizar_fecha_tabla(carpeta, archivos, mapa_columnas, "FECHATRASLADO5")


[FECHATRASLADO5] Iniciando escaneo de calidad temporal...
Análisis guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de fechas\fechatraslado5_analisis_fechas.csv


,Año,Columna,Formato,Total,NULL reales,% NULL,Inválidos,% Inválidos,% Válidos,Año min,Año max,Mes min,Mes max,Día min,Día max
0,2019,FECHATRASLADO5,yyyy-mm-dd,1151398,1149480,99.83,0,0.0,0.17,2015,2019,1,12,1,31
1,2020,FECHATRASLADO5,yyyy-mm-dd,781911,779549,99.70,0,0.0,0.30,2011,2020,1,12,1,31
2,2021,FECHATRASLADO5,yyyy-mm-dd,816909,814360,99.69,0,0.0,0.31,2015,2021,1,12,1,31
3,2022,FECHATRASLADO5,yyyy-mm-dd,932837,930673,99.77,0,0.0,0.23,2018,2022,1,12,1,31
4,2023,FECHATRASLADO5,dd-mm-yyyy,1039587,1037308,99.78,0,0.0,0.22,2022,2023,1,12,1,31
5,2024,FECHATRASLADO5,yyyy-mm-dd,1085813,1083337,99.77,0,0.0,0.23,2020,2024,1,12,1,31


In [9]:
analizar_fecha_tabla(carpeta, archivos, mapa_columnas, "FECHATRASLADO6")


[FECHATRASLADO6] Iniciando escaneo de calidad temporal...
Análisis guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de fechas\fechatraslado6_analisis_fechas.csv


,Año,Columna,Formato,Total,NULL reales,% NULL,Inválidos,% Inválidos,% Válidos,Año min,Año max,Mes min,Mes max,Día min,Día max
0,2019,FECHATRASLADO6,yyyy-mm-dd,1151398,1150562,99.93,0,0.0,0.07,2017,2019,1,12,1,31
1,2020,FECHATRASLADO6,yyyy-mm-dd,781911,780846,99.86,0,0.0,0.14,2011,2020,1,12,1,31
2,2021,FECHATRASLADO6,yyyy-mm-dd,816909,815816,99.87,0,0.0,0.13,2015,2021,1,12,1,31
3,2022,FECHATRASLADO6,yyyy-mm-dd,932837,931801,99.89,0,0.0,0.11,2020,2022,1,12,1,31
4,2023,FECHATRASLADO6,dd-mm-yyyy,1039587,1038566,99.90,0,0.0,0.10,2022,2023,1,12,1,31
5,2024,FECHATRASLADO6,yyyy-mm-dd,1085813,1084664,99.89,0,0.0,0.11,2022,2024,1,12,1,31


In [10]:
analizar_fecha_tabla(carpeta, archivos, mapa_columnas, "FECHATRASLADO7")


[FECHATRASLADO7] Iniciando escaneo de calidad temporal...
Análisis guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de fechas\fechatraslado7_analisis_fechas.csv


,Año,Columna,Formato,Total,NULL reales,% NULL,Inválidos,% Inválidos,% Válidos,Año min,Año max,Mes min,Mes max,Día min,Día max
0,2019,FECHATRASLADO7,yyyy-mm-dd,1151398,1150986,99.96,0,0.0,0.04,2017,2019,1,12,1,31
1,2020,FECHATRASLADO7,yyyy-mm-dd,781911,781414,99.94,0,0.0,0.06,2013,2020,1,12,1,31
2,2021,FECHATRASLADO7,yyyy-mm-dd,816909,816356,99.93,0,0.0,0.07,2015,2021,1,12,1,31
3,2022,FECHATRASLADO7,yyyy-mm-dd,932837,932320,99.94,0,0.0,0.06,2021,2022,1,12,1,31
4,2023,FECHATRASLADO7,dd-mm-yyyy,1039587,1039074,99.95,0,0.0,0.05,2022,2023,1,12,1,31
5,2024,FECHATRASLADO7,yyyy-mm-dd,1085813,1085232,99.95,0,0.0,0.05,2022,2024,1,12,1,31


In [11]:
analizar_fecha_tabla(carpeta, archivos, mapa_columnas, "FECHATRASLADO8")


[FECHATRASLADO8] Iniciando escaneo de calidad temporal...
Análisis guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de fechas\fechatraslado8_analisis_fechas.csv


,Año,Columna,Formato,Total,NULL reales,% NULL,Inválidos,% Inválidos,% Válidos,Año min,Año max,Mes min,Mes max,Día min,Día max
0,2019,FECHATRASLADO8,yyyy-mm-dd,1151398,1151175,99.98,0,0.0,0.02,2017,2019,1,12,1,31
1,2020,FECHATRASLADO8,yyyy-mm-dd,781911,781653,99.97,0,0.0,0.03,2013,2020,1,12,1,31
2,2021,FECHATRASLADO8,yyyy-mm-dd,816909,816614,99.96,0,0.0,0.04,2015,2021,1,12,1,31
3,2022,FECHATRASLADO8,yyyy-mm-dd,932837,932540,99.97,0,0.0,0.03,2021,2022,1,12,1,31
4,2023,FECHATRASLADO8,dd-mm-yyyy,1039587,1039285,99.97,0,0.0,0.03,2022,2023,1,12,1,31
5,2024,FECHATRASLADO8,yyyy-mm-dd,1085813,1085495,99.97,0,0.0,0.03,2022,2024,1,12,1,31


In [12]:
analizar_fecha_tabla(carpeta, archivos, mapa_columnas, "FECHATRASLADO9")


[FECHATRASLADO9] Iniciando escaneo de calidad temporal...
Análisis guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de fechas\fechatraslado9_analisis_fechas.csv


,Año,Columna,Formato,Total,NULL reales,% NULL,Inválidos,% Inválidos,% Válidos,Año min,Año max,Mes min,Mes max,Día min,Día max
0,2019,FECHATRASLADO9,yyyy-mm-dd,1151398,1151271,99.99,0,0.0,0.01,2017,2019,1,12,1,31
1,2020,FECHATRASLADO9,yyyy-mm-dd,781911,781761,99.98,0,0.0,0.02,2014,2020,1,12,1,30
2,2021,FECHATRASLADO9,yyyy-mm-dd,816909,816740,99.98,0,0.0,0.02,2016,2021,1,12,1,31
3,2022,FECHATRASLADO9,yyyy-mm-dd,932837,932666,99.98,0,0.0,0.02,2021,2022,1,12,1,31
4,2023,FECHATRASLADO9,dd-mm-yyyy,1039587,1039408,99.98,0,0.0,0.02,2022,2023,1,12,1,31
5,2024,FECHATRASLADO9,yyyy-mm-dd,1085813,1085636,99.98,0,0.0,0.02,2022,2024,1,12,1,31


### **3. Fecha de Alta (FECHAALTA)**
- **Tipo de variable**: Hospitalaria.
- **Descripción**: Fecha de cierre del episodio asistencial (por alta médica, derivación o fallecimiento).
- **Análisis de Calidad**: Excelente (0% de nulos reales), sujeto al mismo quiebre de formato en 2023 que la fecha de ingreso.
- **Veredicto**: **Descartar columna cruda y Transformar en Variable Derivada (`DIAS_ESTADIA`)**. La fecha de cierre, por sí sola, no es un descriptor del perfil del paciente. Sin embargo, al calcular la diferencia entre `FECHAALTA` y `FECHA_INGRESO`, se obtiene la variable numérica `DIAS_ESTADIA` (o *Length of Stay - LOS*). Esta métrica es uno de los indicadores subyacentes más universales del consumo de recursos y severidad hospitalaria, siendo esencial para los análisis de asociaciones estadísticas y caracterización de subgrupos oncológicos prolongados.

In [13]:
analizar_fecha_tabla(carpeta, archivos, mapa_columnas, "FECHAALTA")


[FECHAALTA] Iniciando escaneo de calidad temporal...
Análisis guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de fechas\fechaalta_analisis_fechas.csv


,Año,Columna,Formato,Total,NULL reales,% NULL,Inválidos,% Inválidos,% Válidos,Año min,Año max,Mes min,Mes max,Día min,Día max
0,2019,FECHAALTA,yyyy-mm-dd,1151398,18,0.0,0,0.0,100.0,2019,2019,1,12,1,31
1,2020,FECHAALTA,yyyy-mm-dd,781911,0,0.0,0,0.0,100.0,2020,2020,1,12,1,31
2,2021,FECHAALTA,yyyy-mm-dd,816909,0,0.0,0,0.0,100.0,2021,2021,1,12,1,31
3,2022,FECHAALTA,yyyy-mm-dd,932837,0,0.0,0,0.0,100.0,2022,2022,1,12,1,31
4,2023,FECHAALTA,dd-mm-yyyy,1039587,0,0.0,0,0.0,100.0,2023,2023,1,12,1,31
5,2024,FECHAALTA,yyyy-mm-dd,1085813,0,0.0,0,0.0,100.0,2024,2024,1,12,1,31


### **4. Fecha de Procedimiento (FECHAPROCEDIMIENTO1)**
- **Tipo de variable**: Hospitalaria.
- **Descripción**: Fecha teórica del primer procedimiento documentado.
- **Análisis de Calidad**: Crítica (100% nulos reales).
- **Veredicto**: **Descartar completamente**. La columna está vacía en todos los archivos anuales analizados (2019-2024), por lo que carece de cualquier utilidad matemática o analítica.

In [14]:
analizar_fecha_tabla(carpeta, archivos, mapa_columnas, "FECHAPROCEDIMIENTO1")


[FECHAPROCEDIMIENTO1] Iniciando escaneo de calidad temporal...
Análisis guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de fechas\fechaprocedimiento1_analisis_fechas.csv


,Año,Columna,Formato,Total,NULL reales,% NULL,Inválidos,% Inválidos,% Válidos,Año min,Año max,Mes min,Mes max,Día min,Día max
0,2019,FECHAPROCEDIMIENTO1,yyyymmdd,1151398,1151389,100.0,9,0.0,0.0,None,None,None,None,None,None
1,2020,FECHAPROCEDIMIENTO1,desconocido,781911,781911,100.0,0,0.0,0.0,None,None,None,None,None,None
2,2021,FECHAPROCEDIMIENTO1,desconocido,816909,816909,100.0,0,0.0,0.0,None,None,None,None,None,None
3,2022,FECHAPROCEDIMIENTO1,desconocido,932837,932837,100.0,0,0.0,0.0,None,None,None,None,None,None
4,2023,FECHAPROCEDIMIENTO1,desconocido,1039587,1039587,100.0,0,0.0,0.0,None,None,None,None,None,None
5,2024,FECHAPROCEDIMIENTO1,desconocido,1085813,1085813,100.0,0,0.0,0.0,None,None,None,None,None,None


### **5. Fecha de Intervención (FECHAINTERV1)**
- **Tipo de variable**: Hospitalaria.
- **Descripción**: Fecha de la primera intervención o cirugía del paciente.
- **Análisis de Calidad**: Regular a Deficiente. Presenta una densidad baja (entre 48% y 50% de nulos), correspondiente a la fracción de pacientes que no requirieron intervención quirúrgica. Más grave aún, presenta outliers e inconsistencias lógicas severas, como registros del año "2000" o "2090" en admisiones que ocurrieron en 2020 o 2023, y casi un 2% de formatos totalmente inválidos en 2024.
- **Veredicto**: **Descartar**. Aunque en estudios de caracterización retrospectiva es válido incluir eventos evolutivos, el alto nivel de corrupción (años imposibles) y la baja densidad de esta columna introducirían ruido excesivo. El valor clínico de la intervención quirúrgica ya está rescatado de forma más estable y confiable mediante la variable binaria `USO_PABELLON` y la variable derivada `PROCEDIMIENTO_QUIRURGICO_PRINCIPAL`.

In [15]:
analizar_fecha_tabla(carpeta, archivos, mapa_columnas, "FECHAINTERV1")


[FECHAINTERV1] Iniciando escaneo de calidad temporal...
Análisis guardado en: ../../Resultados/Resultados (etapa 1 y 2)/Análisis de fechas\fechainterv1_analisis_fechas.csv


,Año,Columna,Formato,Total,NULL reales,% NULL,Inválidos,% Inválidos,% Válidos,Año min,Año max,Mes min,Mes max,Día min,Día max
0,2019,FECHAINTERV1,yyyy-mm-dd,1151398,562241,48.83,0,0.00,51.17,1911,2020,1,12,1,31
1,2020,FECHAINTERV1,yyyy-mm-dd,781911,385855,49.35,5,0.00,50.65,1948,2052,1,12,1,31
2,2021,FECHAINTERV1,yyyy-mm-dd,816909,410033,50.19,10,0.00,49.81,2011,2090,1,12,1,31
3,2022,FECHAINTERV1,yyyy-mm-dd,932837,467603,50.13,0,0.00,49.87,1902,2032,1,12,1,31
4,2023,FECHAINTERV1,dd-mm-yyyy,1039587,510774,49.13,0,0.00,50.87,2002,2090,1,12,1,31
5,2024,FECHAINTERV1,yyyy-mm-dd,1085813,537320,49.49,19669,1.81,48.70,2020,2024,1,12,1,31
